# الفصل 57: المراقبة والتشغيل: Observability

> "ما لا يُقاس لا يُدار".

- **الجزء:** 11 — النشر والتشغيل
- **الفصل:** 57
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 56: البنية التحتية والتوسّع
- **الوسوم:** Metrics، Logging، Tracing، OpenTelemetry

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch57-observability.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** opentelemetry-api, opentelemetry-sdk, prometheus-client
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("opentelemetry", "opentelemetry-api opentelemetry-sdk"),
    ("prometheus_client", "prometheus-client")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch57-observability.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch57-observability.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> "ما لا يُقاس لا يُدار".

> **بيانات الفصل**
>
> - **الجزء:** 11 — النشر والتشغيل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 56: البنية التحتية والتوسّع


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم الأعمدة الثلاثة للمراقبة (Logs, Metrics, Traces)
- فهم لوحات المراقبة والتنبيهات (Dashboards & Alerting)
- فهم Metrics


## 1. لماذا يهمّك هذا الموضوع؟

"ما لا يُقاس لا يُدار". نظام LLM إنتاجي بدون مراقبة شاملة هو صندوق أسود خطير — لن تعرف أنه يتدهور حتى يشتكي المستخدمون، ولن تعرف السبب الجذري حتى يفوت الأوان.


## 2. الخريطة الذهنية


```text
مراقبة LLM
├── التتبع (Traces)
│   ├── Trace ID لكل طلب
│   ├── سلسلة الاستدعاءات (RAG/أدوات/نموذج)
│   └── زمن كل مرحلة
├── المقاييس (Metrics)
│   ├── TTFT / TPOT / p95
│   ├── Throughput / Error rate
│   ├── Tokens / Cost
│   └── Cache hit rate
├── السجلات (Logs)
│   ├── منظَّم: JSON
│   ├── إخفاء PII
│   └── عينة محفوظة
├── جودة المخرجات
│   ├── Faithfulness
│   ├── Refusal rate
│   ├── Toxicity
│   └── User feedback
└── التنبيهات
    ├── SLO burn rate
    ├── قفزة في التكلفة
    └── انخفاض الجودة
```


## 3. الشرح المتدرج

### 🟢 أ) الأعمدة الثلاثة للمراقبة (Logs, Metrics, Traces)


In [ ]:
import time
import logging
import uuid
from dataclasses import dataclass, asdict
from prometheus_client import Counter, Histogram, Gauge

# ==================== 1. Metrics (مقاييس كمّية مُجمَّعة) ====================
REQUEST_COUNT = Counter('llm_requests_total', 'إجمالي الطلبات', ['model', 'status'])
REQUEST_LATENCY = Histogram('llm_request_duration_seconds', 'زمن معالجة الطلب',
                              buckets=[0.1, 0.5, 1, 2, 5, 10, 30])
TOKENS_GENERATED = Counter('llm_tokens_generated_total', 'إجمالي الرموز المُولَّدة')
TIME_TO_FIRST_TOKEN = Histogram('llm_ttft_seconds', 'الزمن حتى أول رمز (مهم جدًا لتجربة المستخدم)')
ACTIVE_REQUESTS = Gauge('llm_active_requests', 'عدد الطلبات النشطة حاليًا')
GPU_MEMORY_USAGE = Gauge('llm_gpu_memory_used_bytes', 'استهلاك ذاكرة GPU')

# ==================== 2. Structured Logging (سجلات مُهيكَلة قابلة للاستعلام) ====================
@dataclass
class InferenceLogEntry:
    request_id: str
    timestamp: float
    model_name: str
    prompt_tokens: int
    completion_tokens: int
    total_latency_ms: float
    time_to_first_token_ms: float
    user_id: str
    status: str
    error_message: str = None

logger = logging.getLogger("llm_service")

def log_inference_structured(entry: InferenceLogEntry):
    """تسجيل بصيغة JSON مُهيكَلة — قابلة للاستعلام في Elasticsearch/Loki/CloudWatch"""
    logger.info(json.dumps(asdict(entry), ensure_ascii=False))

# ==================== 3. Distributed Tracing (تتبّع موزّع عبر خدمات متعددة) ====================
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

tracer = trace.get_tracer("llm-service")

class InstrumentedLLMService:
    """خدمة استدلال كاملة مع مراقبة شاملة مدمجة"""
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def generate_with_full_observability(self, prompt: str, user_id: str) -> dict:
        request_id = str(uuid.uuid4())
        start_time = time.time()
        ACTIVE_REQUESTS.inc()

        with tracer.start_as_current_span("llm_inference") as span:
            span.set_attribute("request_id", request_id)
            span.set_attribute("user_id", user_id)
            span.set_attribute("prompt_length", len(prompt))

            try:
                with tracer.start_as_current_span("tokenization"):
                    input_ids = self.tokenizer(prompt, return_tensors="pt").input_ids
                    prompt_tokens = input_ids.shape[1]

                with tracer.start_as_current_span("generation"):
                    first_token_time = None
                    output_tokens = []
                    for token in self._stream_generate(input_ids):
                        if first_token_time is None:
                            first_token_time = time.time()
                            ttft = first_token_time - start_time
                            TIME_TO_FIRST_TOKEN.observe(ttft)
                        output_tokens.append(token)

                total_latency = time.time() - start_time
                REQUEST_LATENCY.observe(total_latency)
                REQUEST_COUNT.labels(model="llama-3-8b", status="success").inc()
                TOKENS_GENERATED.inc(len(output_tokens))

                log_inference_structured(InferenceLogEntry(
                    request_id=request_id, timestamp=start_time, model_name="llama-3-8b",
                    prompt_tokens=prompt_tokens, completion_tokens=len(output_tokens),
                    total_latency_ms=total_latency * 1000,
                    time_to_first_token_ms=(first_token_time - start_time) * 1000,
                    user_id=user_id, status="success"
                ))
                span.set_status(Status(StatusCode.OK))
                return {"request_id": request_id, "response": self.tokenizer.decode(output_tokens)}

            except Exception as e:
                REQUEST_COUNT.labels(model="llama-3-8b", status="error").inc()
                log_inference_structured(InferenceLogEntry(
                    request_id=request_id, timestamp=start_time, model_name="llama-3-8b",
                    prompt_tokens=0, completion_tokens=0, total_latency_ms=(time.time()-start_time)*1000,
                    time_to_first_token_ms=0, user_id=user_id, status="error", error_message=str(e)
                ))
                span.set_status(Status(StatusCode.ERROR, str(e)))
                raise
            finally:
                ACTIVE_REQUESTS.dec()

    def _stream_generate(self, input_ids): ...  # توليد تدفقي token-by-token


> 💡 **مقياس حاسم غالبًا ما يُهمَل: Time To First Token (TTFT)**. المستخدمون أكثر حساسية لزمن **الانتظار قبل ظهور أول كلمة** منهم لزمن اكتمال الاستجابة بأكملها — تطبيق يُظهر النص فورًا (streaming) بـ TTFT منخفض يبدو أسرع بكثير من تطبيق ينتظر الاستجابة كاملة حتى لو كان الزمن الكلي متطابقًا.

### 🟡 ب) لوحات المراقبة والتنبيهات (Dashboards & Alerting)


In [ ]:
"""
مثال قواعد تنبيه Prometheus/Alertmanager لخدمة LLM إنتاجية
"""
ALERTING_RULES_YAML = """
groups:
- name: llm_service_alerts
  rules:
  - alert: HighLatencyP99
    expr: histogram_quantile(0.99, llm_request_duration_seconds_bucket) > 10
    for: 5m
    annotations:
      summary: "زمن الاستجابة عند p99 يتجاوز 10 ثوانٍ لأكثر من 5 دقائق"

  - alert: HighErrorRate
    expr: rate(llm_requests_total{status="error"}[5m]) / rate(llm_requests_total[5m]) > 0.05
    for: 2m
    annotations:
      summary: "معدل الأخطاء يتجاوز 5% — تحقق فورًا"

  - alert: GPUMemoryNearCapacity
    expr: llm_gpu_memory_used_bytes / llm_gpu_memory_total_bytes > 0.95
    for: 1m
    annotations:
      summary: "ذاكرة GPU قريبة من الامتلاء الكامل — خطر OOM وشيك"

  - alert: QueueBacklogGrowing
    expr: llm_active_requests > 50
    for: 3m
    annotations:
      summary: "تراكم طابور الطلبات — قد تحتاج Autoscaling يدويًا أو فحص عنق الزجاجة"
"""


### 🔴 ج) طبقة التعمّق: مراقبة جودة المخرجات (وليس فقط الأداء التقني)

المراقبة التقليدية (latency، throughput، أخطاء HTTP) **لا تكشف** تدهور **جودة** مخرجات النموذج نفسه — نموذج قد يستجيب بسرعة وبدون أخطاء تقنية، لكن بإجابات رديئة أو منحرفة عن الغرض.


In [ ]:
class OutputQualityMonitor:
    """
    مراقبة مستمرة لجودة المخرجات في الإنتاج، تكملة لمراقبة الأداء التقني
    (يبني على مفاهيم الفصل 46 — Continuous Evaluation، لكن بتركيز على الإنتاج الحي)
    """
    def __init__(self, safety_classifier, judge_client, sample_rate=0.02):
        self.safety_classifier = safety_classifier
        self.judge_client = judge_client
        self.sample_rate = sample_rate   # عيّنة من الطلبات فقط (لتقليل التكلفة)
        self.quality_history = []

    def monitor_response(self, prompt: str, response: str, request_id: str):
        # 1. فحص فوري وزهيد لكل طلب: اكتشاف أنماط مقلقة سريعة (طول غير طبيعي، تكرار)
        if self._detect_degenerate_output(response):
            self._raise_alert("DEGENERATE_OUTPUT", request_id)

        # 2. فحص أعمق (أغلى) لعيّنة فقط من الطلبات
        import random
        if random.random() < self.sample_rate:
            safety_result = self.safety_classifier(response)
            if safety_result["is_harmful"]:
                self._raise_alert("HARMFUL_CONTENT_DETECTED", request_id)

            quality_score = llm_judge_evaluate(self.judge_client, prompt, response)
            self.quality_history.append({"request_id": request_id, "score": quality_score["overall"]})

            # كشف انجراف الجودة (Quality Drift) عبر نافذة متحركة
            if len(self.quality_history) >= 100:
                recent_avg = sum(q["score"] for q in self.quality_history[-100:]) / 100
                if recent_avg < 6.0:   # عتبة جودة دنيا
                    self._raise_alert("QUALITY_DRIFT_DETECTED", request_id,
                                        details=f"متوسط الجودة الأخير: {recent_avg}")

    def _detect_degenerate_output(self, response: str) -> bool:
        """كشف سريع لمخرجات متدهورة شائعة: تكرار لا نهائي، فراغ، طول شاذ"""
        words = response.split()
        if len(words) == 0:
            return True
        if len(set(words[-20:])) < 5 and len(words) > 20:   # تكرار شديد في آخر الاستجابة
            return True
        return False

    def _raise_alert(self, alert_type: str, request_id: str, details: str = ""):
        logger.warning(f"⚠️ تنبيه جودة [{alert_type}] للطلب {request_id}: {details}")


## 4. مثال محلول خطوة بخطوة

**الطلب:** إجابة RAG لمستخدم في تطبيق دعم.

**ميزانية زمن الاستجابة p95:**

| المرحلة | الزمن (ms) | النسبة |
|---|---:|---:|
| الشبكة/البوابة | 70 | 4% |
| الانتظار في الطابور | 200 | 11% |
| Prefill (معالجة الـ prompt) | 350 | 19% |
| Decode (توليد 300 token) | 1,200 | 64% |
| المعالجة اللاحقة | 50 | 3% |
| **الإجمالي** | **1,870** | **100%** |

**التشخيص:** Decode هو العنصر الأكبر. خيارات التحسين:
- تقليل `max_tokens` إلى 200 يوفّر نحو 400ms.
- استخدام نموذج أصغر للطلبات البسيطة يقلّل TPOT.
- Speculative decoding قد يخفض Decode 30-50% إذا كان معدل القبول مرتفعًا.
- تحسين الطابور (نسخ إضافية) يقلّل 200ms.

**لوحة المراقبة المقترحة:**

| اللوحة | المقاييس |
|---|---|
| الأداء | TTFT p50/p95، TPOT، throughput، queue depth |
| الموثوقية | error rate، timeout rate، OOM count |
| التكلفة | tokens/request، cost/request، cost/day |
| الجودة | faithfulness، refusal rate، user thumbs-down |
| الأمان | blocked requests، injection attempts |

**الخلاصة:** لا تنظر إلى متوسط الزمن؛ p95 يكشف تجربة المستخدم الحقيقية. وابدأ بالمرحلة الأكبر في الميزانية.


## 5. نصائح وحيل

> 💡 **اربط كل سجل بـ Trace ID**: بدون معرف موحّد، تصبح debugging قصة تخمين. مرّر الـ ID عبر API، RAG، الأدوات، والنموذج.

> 💡 **سجّل الملخص لا النص الكامل**: احفظ hash/طول/لغة الـ prompt، وعينة صغيرة مشفرة للمراجعة. لا تخزّن PII في السجلات.

> 💡 **استخدم SLOs لا مجرد تنبيهات**: «p95 < 2s» و«error rate < 1%» أهداف واضحة. نبّه على معدل استهلاك الميزانية (burn rate) لا على لحظة تجاوز واحدة.

> 💡 **راقب الجودة في الإنتاج**: عينة 1% من الطلبات تُقيَّم آليًا (LLM judge) وبشريًا أسبوعيًا. الأداء التقني لا يكشف هلوسة أو انحرافًا.

> 💡 **اختبر التنبيهات**: أرسل تنبيهًا تجريبيًا وتأكد أنه يصل، وأنه قابل للتنفيذ. التنبيه الذي لا يُفعّل شيئًا هو ضوضاء.


## 6. أخطاء شائعة ومصائد

> ⚠️ **مراقبة الأداء فقط**: خدمة سريعة تُنتج إجابات خاطئة هي خدمة سيئة. أضف مقاييس جودة.

> ⚠️ **تسجيل prompts كاملة مع PII**: مخاطر خصوصية وامتثال. استخدم إخفاءً أو تجزئة، وسياسة احتفاظ واضحة.

> ⚠️ **الاعتماد على المتوسطات**: متوسط 500ms قد يخفي p99 عند 8 ثوانٍ. استخدم percentiles دائمًا.

> ⚠️ **إغراق الفريق بالتنبيهات**: كل انحراف صغير يُطلق تنبيهًا. اضبط العتبات وادمج التنبيهات المتشابهة.

> ⚠️ **عدم ربط التكلفة بالطلب**: بدون cost attribution لا تعرف أي ميزة أو مستأجر يستهلك الميزانية. أضف وسومًا لكل طلب.


## 7. جدول مقارنة

| المستوى | ماذا يراقب | أدوات نموذجية |
|---|---|---|
| **البنية التحتية** | CPU، GPU، ذاكرة، شبكة | Prometheus + Grafana، NVIDIA DCGM |
| **الأداء التقني** | latency، throughput، TTFT، معدل الأخطاء | Prometheus، Datadog، New Relic |
| **التتبّع الموزّع** | مسار الطلب عبر خدمات متعددة | OpenTelemetry، Jaeger |
| **جودة المخرجات** | سلامة، صدق، اتّساق، انجراف الجودة | أنظمة مخصصة (الفصل 46/57) |
| **تجربة المستخدم** | رضا، معدل إعادة الصياغة، معدل التخلي | Analytics مخصصة + Implicit feedback |


## 8. ملخص أفضل الممارسات

- [ ] قياس TTFT بشكل منفصل عن زمن الاستجابة الكلي
- [ ] تنبيهات آلية لمعدل الأخطاء، latency p99، وامتلاء ذاكرة GPU
- [ ] تتبّع موزّع (tracing) لكل مراحل معالجة الطلب (tokenization → retrieval → generation)
- [ ] مراقبة جودة مخرجات مستمرة بعيّنة عشوائية (ليس فقط مراقبة تقنية)
- [ ] سجلات مُهيكَلة (structured logs) قابلة للاستعلام الفوري عند التحقيق في حادثة

---


## 9. أسئلة مقابلات

**سؤال 1: ما أهم مقاييس مراقبة خدمة LLM؟**

> **إجابة نموذجية:** على مستوى الأداء: TTFT، TPOT، p95 latency، throughput، ومعدل الأخطاء. على مستوى التكلفة: tokens/request وcost/request. على مستوى الجودة: faithfulness، refusal rate، ورضا المستخدم. على مستوى الأمان: الطلبات المحجوبة ومحاولات الحقن. المزيج هو ما يعطي صورة كاملة؛ أي مجموعة منفردة مضللة.

**سؤال 2: كيف تكتشف انحراف الجودة (Quality Drift)؟**

> **إجابة نموذجية:** أبني مجموعة تقييم ثابتة تُشغَّل دوريًا، وأقارن النتائج بالإصدار السابق. في الإنتاج، أُقيّم عينة من الطلبات آليًا (LLM judge مع rubric) وأراجعها بشريًا. أتابع مؤشرات غير مباشرة: زيادة الرفض، زيادة التصعيد، انخفاض thumbs-up، وتغير توزيع الأسئلة. أي قفزة تُفعّل تحقيقًا.

**سؤال 3: كيف تتتبع طلبًا عبر RAG وأدوات متعددة؟**

> **إجابة نموذجية:** أُنشئ Trace ID عند البوابة وأمرّره في كل استدعاء. كل مرحلة (استرجاع، إعادة ترتيب، استدعاء أداة، توليد) تُسجّل span مع بداية/نهاية، مدخلات ملخصة، ومخرجات ملخصة. أستخدم OpenTelemetry أو منصة مثل Langfuse، وأربط الـ spans في شجرة واحدة. هذا يكشف أين ذهب الزمن وأين حدث الخطأ.


## 10. تمرين عملي

**الهدف:** تصميم لوحة مراقبة لروبوت دعم عربي.

**الخطوات:**
1. حدّد 8 مؤشرات على الأقل موزعة على الأداء والجودة والتكلفة والأمان.
2. لكل مؤشر: مصدر البيانات، التحديث، والعتبة.
3. صمّم 3 تنبيهات فقط: p95 latency، error rate، وانخفاض thumbs-up.
4. حدّد سياسة الاحتفاظ بالسجلات وإخفاء PII.
5. ارسم مخطط لوحة (ورقيًا أو بـ Mermaid).
6. اكتب إجراءً واحدًا لكل تنبيه: من يستجيب؟ وماذا يفعل؟

**معيار النجاح:** لوحة يمكن لفريق المناوبة فهمها في 30 ثانية، وتنبيهات قابلة للتنفيذ.


## 11. مراجع للتعمّق

- OpenTelemetry Documentation. opentelemetry.io/docs.
- Beyer, B., et al. (2016). *Site Reliability Engineering*. O'Reilly.
- Langfuse Documentation. langfuse.com/docs.
- Arize. *Phoenix: Open-Source LLM Observability*. docs.arize.com/phoenix.
- Zheng, L., et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النشر والتشغيل / الفصل 57: المراقبة والتشغيل: Observability.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
